# RAG poisoning detection: Colab runner
Run the cells **top to bottom, one at a time**. Before starting: *Runtime → Change runtime type → T4 GPU*.
Everything important is saved to your Google Drive (`MyDrive/ragdet`), so a disconnect does not lose finished stages.
**Status: the GPU code has never been run before. Expect to hit and fix a few errors on the first tiny run.**

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Connect Google Drive and set paths
The model download (~6 GB) and data go to Drive so you only download them once. Make sure your Drive has ~10 GB free.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ["RAGDET_ROOT"] = "/content/drive/MyDrive/ragdet"
os.environ["HF_HOME"] = "/content/drive/MyDrive/ragdet/hf_cache"
os.environ["PYTHONPATH"] = "/content/rag-poison-detect/src"
os.makedirs(os.environ["RAGDET_ROOT"], exist_ok=True)
print("paths set")

## 3. Get the code from GitHub (and install what's missing)

In [ ]:
%cd /content
!git clone https://github.com/krishhhna7/rag-poison-detect.git || (cd rag-poison-detect && git pull)
%cd /content/rag-poison-detect
!pip -q install transformers accelerate pyyaml tqdm

## 4. Run the unit tests (should say `22 passed, 1 skipped`)

In [ ]:
!python -m pytest -q

## 5. Download the NQ data (one time only)
Corpus from BEIR plus the PoisonedRAG question file. This can take several minutes.

In [ ]:
!bash scripts/download_data.sh nq

## 6. Check the files look as expected
We *assumed* the standard BEIR layout. If any of these commands fail or look different, stop and send the output.

In [ ]:
!ls -la $RAGDET_ROOT/data/nq $RAGDET_ROOT/data/nq/qrels
!head -c 500 $RAGDET_ROOT/data/nq/corpus.jsonl
!echo
!head -3 $RAGDET_ROOT/data/nq/qrels/test.tsv

## 7. TINY run first (10 questions)
This finds bugs cheaply and tells us the real speed. Use a separate run name so it does not mix with the full run.
If a cell errors, copy the **whole** error text and send it.

In [ ]:
!python -m ragdet.cli attack   --config configs/small_gpu.yaml --set data.n_targets=10 run_name=colab_tiny

In [ ]:
!python -m ragdet.cli features --config configs/small_gpu.yaml --set data.n_targets=10 run_name=colab_tiny

In [ ]:
!python -m ragdet.cli a2       --config configs/small_gpu.yaml --set data.n_targets=10 run_name=colab_tiny

In [ ]:
!python -m ragdet.cli evaluate --config configs/small_gpu.yaml --set data.n_targets=10 run_name=colab_tiny

## 8. FULL run (100 questions)
Only do this after the tiny run worked. If Colab disconnects, reconnect, re-run cells 2, 3 and then continue from the stage that did not finish.
If you change attack settings, use a new `run_name` (finished stages are cached).

In [ ]:
!python -m ragdet.cli attack   --config configs/small_gpu.yaml

In [ ]:
!python -m ragdet.cli features --config configs/small_gpu.yaml

In [ ]:
!python -m ragdet.cli a2       --config configs/small_gpu.yaml

In [ ]:
!python -m ragdet.cli evaluate --config configs/small_gpu.yaml

## 9. Look at the results
Files are in Drive: `MyDrive/ragdet/runs/<run_name>/`. Download `results_summary.csv` and `results_raw.csv` and commit them to GitHub.

In [ ]:
import pandas as pd
df = pd.read_csv(os.environ["RAGDET_ROOT"] + "/runs/nq_qwen3b_contriever/results_summary.csv")
df[df.metric.isin(["auroc","tpr","fpr","query_tpr"])]